# gnomAD-common control fetch (run in Google Colab)

Fetches gnomAD v4.1 common SNVs (AF >= 0.05) in the regulatory neighbourhoods of the ClinVar pathogenic variants, to serve as frequency-based controls for the composition-confound audit.

**Why Colab:** gnomAD lives on Google Cloud Storage; Colab streams it fast (the same remote tabix stalls on a home connection). No GPU needed — set Runtime to CPU.

**Steps:** Run all cells. It writes `gnomad_common_controls.csv` and downloads it. Send that file back; the rest of the analysis (GPN-MSA + composition scoring, recovery comparison) is done locally.

In [ ]:
!pip -q install pysam pandas
import pysam, pandas as pd, numpy as np, random

In [ ]:
# pathogenic positions (public, from the repo)
URL = 'https://raw.githubusercontent.com/rikhinkavuru/composition-confound-vep/main/notebooks/clinvar_pathogenic_positions.csv'
path = pd.read_csv(URL, dtype={'chrom': str})
path_pos = set(zip(path.chrom, path.pos))
print('pathogenic positions:', len(path), '| chroms:', sorted(path.chrom.unique()))

In [ ]:
GN = ('https://storage.googleapis.com/gcp-public-data--gnomad/release/4.1/vcf/genomes/'
      'gnomad.genomes.v4.1.sites.chr{c}.vcf.bgz')
PAD, AF_MIN, MERGE = 1000, 0.05, 20000

def merge_spans(pos):
    spans = []
    for p in sorted(pos):
        s, e = p - PAD, p + PAD
        if spans and s <= spans[-1][1] + MERGE:
            spans[-1][1] = max(spans[-1][1], e)
        else:
            spans.append([s, e])
    return spans

rows = []
for c in [str(i) for i in range(1, 23)] + ['X']:
    g = path[path.chrom == c]
    if not len(g):
        continue
    try:
        vf = pysam.VariantFile(GN.format(c=c))
    except Exception as ex:
        print('open fail', c, ex); continue
    got = 0
    for s, e in merge_spans(g.pos.unique()):
        try:
            for r in vf.fetch(f'chr{c}', max(0, s), e):
                af = r.info.get('AF')
                if (af and af[0] and af[0] >= AF_MIN and len(r.ref) == 1 and r.alts
                        and len(r.alts[0]) == 1 and r.ref in 'ACGT' and r.alts[0] in 'ACGT'
                        and (c, r.pos) not in path_pos):
                    rows.append((f'chr{c}', r.pos, r.ref, r.alts[0], round(af[0], 4)))
                    got += 1
        except Exception:
            continue
    print(f'chr{c}: {got} common SNVs')

ctl = pd.DataFrame(rows, columns=['chrom', 'pos', 'ref', 'alt', 'gnomad_af']).drop_duplicates(['chrom', 'pos', 'ref', 'alt'])
print('TOTAL gnomAD-common controls:', len(ctl))
ctl.to_csv('gnomad_common_controls.csv', index=False)

In [ ]:
from google.colab import files
files.download('gnomad_common_controls.csv')